# Betting Data Analysis

In [23]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data.csv')
print(f'Dataset: {df.shape[0]} bets, {df.shape[1]} columns')
df.head()

Dataset: 1117 bets, 77 columns


,Unique Bet Rule 1,Team,Total?,Unnamed: 3,Bet,Book,Date,Market,Other,LineTaken,...,CZR M,CZR O,BM M$,BM O$,DK M,DK O,4C M,4C M$,4C O,4C O$
0,1,Pistons,0,2024-10,Pistons +30.5 L,FWmb,2024-10-26,-1400,630,-500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Pistons,0,2024-10,Pistons +32.5 L,FWmb,2024-10-26,-1450,640,-480,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Pistons,0,2024-10,Pistons +29.5 L,FWmb,2024-10-26,-750,430,-278,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,Clippers,0,2024-10,Clippers ml L,FWmb,2024-10-31,-199,169,-165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,Clippers,0,2024-10,Clippers ml L,BRmb,2024-10-31,-259,198,-200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# --- Data Cleanup ---

# 1. Strip whitespace from column names
df.columns = df.columns.str.strip()

# 2. Convert Date columns to datetime
df['Date'] = pd.to_datetime(df['Date'])
df['Date.1'] = pd.to_datetime(df['Date.1'])

# 3. Drop Time column
df.drop(columns=['Time'], inplace=True)

# 4. Rename Unnamed: 3 to Month
df.rename(columns={'Unnamed: 3': 'Month'}, inplace=True)

# 5. Rename Unnamed: 16 to BetType
df.rename(columns={'Unnamed: 16': 'BetType'}, inplace=True)

# 6. Total? to bool
df['Total?'] = df['Total?'].astype(bool)

# 7. Grade to categorical
df['Grade'] = df['Grade'].astype('category')

# 8. Drop Ignore columns
df.drop(columns=['Ignore', 'Ignore.1'], inplace=True)

# 9. Drop 100% null columns
null_cols = [c for c in df.columns if df[c].isna().all()]
print(f'Dropping {len(null_cols)} fully null columns: {null_cols}')
df.drop(columns=null_cols, inplace=True)

print(f'\nCleaned: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nDtypes:\n{df.dtypes.to_string()}')

Dropping 8 fully null columns: ['CloseLine', 'CloseOther', 'CloseEdge', 'Trueline', '4C M', '4C M$', '4C O', '4C O$']

Cleaned: 1117 rows, 66 columns

Dtypes:
Unique Bet Rule 1             int64
Team                         object
Total?                         bool
Month                        object
Bet                          object
Book                         object
Date                 datetime64[ns]
Market                        int64
Other                         int64
LineTaken                     int64
Edge                        float64
TrueL                       float64
Decim                       float64
Betsize                     float64
Sport                        object
BetType                      object
Opp                          object
Total Stake                 float64
Rich Stake                  float64
Other Stake                 float64
Grade                      category
Net                         float64
ExpWinPlace                 float64
Date.1       

In [25]:
# --- Analysis DataFrame ---

# Filter to Basketball only and 2026 bets
bdf = df[(df['Sport'] == 'Basketball') & (df['Date'].dt.year == 2026)].copy()
print(f'Filtered to Basketball in 2026: {len(bdf)} rows')

# Select and rename columns
bdf = bdf[['Unique Bet Rule 1', 'Team', 'Total?', 'Bet', 'Date', 'Market',
            'Other', 'LineTaken', 'Edge', 'BetType', 'Rich Stake', 'Grade']].copy()

bdf.rename(columns={
    'Unique Bet Rule 1': 'Bet',
    'Bet': 'ExactBet',
    'Total?': 'Total',
    'Rich Stake': 'RichStake',
    'LineTaken': 'LineTaken',
}, inplace=True)

# Convert Edge from decimal to percentage with 1 decimal place
bdf['Edge'] = (bdf['Edge'] * 100).round(1)

# Round RichStake to 2 decimal places to fix floating point precision
bdf['RichStake'] = bdf['RichStake'].round(2)

# Calculate Expected Profit
bdf['ExpProfit'] = (bdf['RichStake'] * bdf['Edge'] / 100).round(0).astype(int)

# Calculate Profit
def calc_profit(row):
    if row['Grade'] == 'P':
        return 0.0
    if row['Grade'] == 'L':
        return -row['RichStake']
    # Grade == W: calculate winnings from american odds
    odds = row['LineTaken']
    stake = row['RichStake']
    if odds > 0:
        return stake * (odds / 100)
    else:
        return stake * (100 / abs(odds))

bdf['Profit'] = bdf.apply(calc_profit, axis=1).astype(int)

bdf.to_csv('clean_data.csv', index=False)
print(f'{len(bdf)} bets | Total Profit: ${bdf["Profit"].sum():,}')
print('Exported to clean_data.csv')
bdf.head(10)

Filtered to Basketball in 2026: 465 rows
465 bets | Total Profit: $-13,882
Exported to clean_data.csv


,Bet,Team,Total,ExactBet,Date,Market,Other,LineTaken,Edge,BetType,RichStake,Grade,ExpProfit,Profit
652,297,Air,False,Air Force +40.5 L,2026-01-24,-139,105,104,10.9,spread,455.0,L,50,-455
653,298,Arizona,False,Arizona -19.5 L,2026-01-24,-175,130,104,21.2,spread,297.5,W,63,309
654,299,Auburn,True,Auburn o141.5 L,2026-01-24,100,-120,125,7.6,total,560.0,W,43,700
655,300,Binghamton,False,Binghamton ml L,2026-01-24,294,-294,400,26.9,moneyline,122.5,L,33,-122
656,301,Boston,False,Boston College +6.5 L,2026-01-24,102,-110,132,12.7,spread,367.5,W,47,485
657,302,Boston,True,Boston Celtics u236.5 L,2026-01-24,-107,-109,123,11.0,total,980.0,W,108,1205
658,302,Boston,True,Boston Celtics u232.5 L,2026-01-24,108,-136,133,6.0,total,735.0,W,44,977
659,303,Bowling,False,Bowling Green +2.5 L,2026-01-24,-106,106,118,12.2,spread,262.5,W,32,309
660,304,Brown,False,Brown ml L,2026-01-24,305,-450,255,-17.7,moneyline,175.0,L,-31,-175
661,305,Brown,True,Brown o131.5 L,2026-01-24,-111,111,-111,0.0,total,350.0,L,0,-350


In [26]:
# --- Merge rows by Bet ID ---

mdf = pd.read_csv('clean_data.csv')

def american_to_decimal(odds):
    """Convert american odds to decimal odds."""
    if odds > 0:
        return (odds / 100) + 1
    elif odds < 0:
        return (100 / abs(odds)) + 1
    else:
        return 2.0  # even money

def decimal_to_american(dec):
    """Convert decimal odds back to american odds."""
    if dec >= 2.0:
        return round((dec - 1) * 100)
    else:
        return round(-100 / (dec - 1))

def median_odds(series):
    """Convert to decimal, take median, convert back to american."""
    decimal_odds = series.apply(american_to_decimal)
    med = decimal_odds.median()
    return decimal_to_american(med)

def grade_agg(grades):
    unique = grades.unique()
    if len(unique) == 1:
        return unique[0]
    return 'mix'

merged = mdf.groupby('Bet').agg(
    Team=('Team', 'first'),
    Total=('Total', 'first'),
    ExactBet=('ExactBet', 'first'),
    Date=('Date', 'first'),
    Market=('Market', median_odds),
    Other=('Other', median_odds),
    LineTaken=('LineTaken', median_odds),
    Edge=('Edge', 'median'),
    BetType=('BetType', 'first'),
    RichStake=('RichStake', 'sum'),
    Grade=('Grade', grade_agg),
    Profit=('Profit', 'sum'),
).reset_index()

# Fix floating point precision
merged['Edge'] = merged['Edge'].round(1)
merged['RichStake'] = merged['RichStake'].round(2)

merged.to_csv('merged_data.csv', index=False)
print(f'Merged: {len(mdf)} rows -> {len(merged)} bets')
print(f'Grade breakdown: {merged["Grade"].value_counts().to_dict()}')
print('Exported to merged_data.csv')
merged.head(10)

Merged: 465 rows -> 304 bets
Grade breakdown: {'L': 147, 'W': 133, 'mix': 23, 'P': 1}
Exported to merged_data.csv


,Bet,Team,Total,ExactBet,Date,Market,Other,LineTaken,Edge,BetType,RichStake,Grade,Profit
0,297,Air,False,Air Force +40.5 L,2026-01-24,-139,105,104,10.9,spread,455.0,L,-455
1,298,Arizona,False,Arizona -19.5 L,2026-01-24,-175,130,104,21.2,spread,297.5,W,309
2,299,Auburn,True,Auburn o141.5 L,2026-01-24,100,-120,125,7.6,total,560.0,W,700
3,300,Binghamton,False,Binghamton ml L,2026-01-24,294,-294,400,26.9,moneyline,122.5,L,-122
4,301,Boston,False,Boston College +6.5 L,2026-01-24,102,-110,132,12.7,spread,367.5,W,485
5,302,Boston,True,Boston Celtics u236.5 L,2026-01-24,101,-121,128,8.5,total,1715.0,W,2182
6,303,Bowling,False,Bowling Green +2.5 L,2026-01-24,-106,106,118,12.2,spread,262.5,W,309
7,304,Brown,False,Brown ml L,2026-01-24,305,-450,255,-17.7,moneyline,175.0,L,-175
8,305,Brown,True,Brown o131.5 L,2026-01-24,-117,106,-102,4.1,total,630.0,L,-630
9,306,California,True,California u148.5 L,2026-01-24,-122,-108,114,10.0,total,262.5,W,299


In [27]:
# --- Performance Analysis by Bucket ---

cdf = pd.read_csv('clean_data.csv')

# Exclude pushes from analysis
cdf = cdf[cdf['Grade'] != 'P'].copy()

def summarize(group):
    """Return performance summary for a group of bets."""
    staked = group['RichStake'].sum()
    exp_profit = group['ExpProfit'].sum()
    return pd.Series({
        'Bets': int(len(group)),
        'Wins': int((group['Grade'] == 'W').sum()),
        'Win%': round((group['Grade'] == 'W').mean() * 100, 1),
        'Staked': int(staked),
        'ExpProfit': int(exp_profit),
        'Profit': int(group['Profit'].sum()),
        'ExpROI%': round(exp_profit / staked * 100, 1),
        'ROI%': round(group['Profit'].sum() / staked * 100, 1),
    })

def int_cols(summary):
    """Cast integer columns."""
    for col in ['Bets', 'Wins', 'Staked', 'ExpProfit', 'Profit']:
        summary[col] = summary[col].astype(int)
    return summary

# --- 1. By Odds Bands (LineTaken) ---
odds_bins = [-999999, -150, -100, 150, 999999]
odds_labels = ['-150 or less', '-150 to -100', '+100 to +150', '+150 or greater']
cdf['OddsBand'] = pd.cut(cdf['LineTaken'], bins=odds_bins, labels=odds_labels)

print('=== Performance by Odds Band ===')
odds_summary = cdf.groupby('OddsBand', observed=False).apply(summarize, include_groups=False).reset_index()
display(int_cols(odds_summary))

# --- 2. By Edge Bands ---
edge_bins = [0, 5, 10, 15, float('inf')]
edge_labels = ['0-5%', '5-10%', '10-15%', '15%+']
cdf['EdgeBand'] = pd.cut(cdf['Edge'], bins=edge_bins, labels=edge_labels)

print('\n=== Performance by Edge Band ===')
edge_summary = cdf.groupby('EdgeBand', observed=False).apply(summarize, include_groups=False).reset_index()
display(int_cols(edge_summary))

# --- 3. By BetType ---
print('\n=== Performance by BetType ===')
core_types = cdf[cdf['BetType'].isin(['moneyline', 'spread', 'total'])]
bettype_summary = core_types.groupby('BetType').apply(summarize, include_groups=False).reset_index()
display(int_cols(bettype_summary))

# --- 4. Grand Totals ---
print('\n=== Grand Totals ===')
totals = summarize(cdf).to_frame().T
display(int_cols(totals))

=== Performance by Odds Band ===


,OddsBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,-150 or less,12,7,58.3,27209,2068,-2,7.6,-0.0
1,-150 to -100,86,50,58.1,107363,10517,-4750,9.8,-4.4
2,+100 to +150,290,145,50.0,197983,22633,-3230,11.4,-1.6
3,+150 or greater,76,23,30.3,35984,6743,-5900,18.7,-16.4



=== Performance by Edge Band ===


,EdgeBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,0-5%,19,11,57.9,24857,720,-4342,2.9,-17.5
1,5-10%,139,72,51.8,127138,10534,3894,8.3,3.1
2,10-15%,164,80,48.8,125927,14964,-9614,11.9,-7.6
3,15%+,116,49,42.2,71972,16523,-1546,23.0,-2.1



=== Performance by BetType ===


,BetType,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,moneyline,119,57,47.9,101368,12742,13406,12.6,13.2
1,spread,191,89,46.6,176906,19123,-26276,10.8,-14.9
2,total,151,78,51.7,88502,9929,-789,11.2,-0.9



=== Grand Totals ===


,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,464,225,48.5,368540,41961,-13882,11.4,-3.8


In [28]:
# --- Performance by Stake Band (Merged Data) ---

mdf = pd.read_csv('merged_data.csv')
mdf = mdf[mdf['Grade'] != 'P'].copy()

# Add ExpProfit (Edge is already a percentage)
mdf['ExpProfit'] = (mdf['RichStake'] * mdf['Edge'] / 100).round(0).astype(int)

stake_bins = [0, 500, 1000, 2000, float('inf')]
stake_labels = ['0-500', '500-1000', '1000-2000', '2000+']
mdf['StakeBand'] = pd.cut(mdf['RichStake'], bins=stake_bins, labels=stake_labels)

def summarize(group):
    staked = group['RichStake'].sum()
    exp_profit = group['ExpProfit'].sum()
    return pd.Series({
        'Bets': int(len(group)),
        'Wins': int((group['Grade'] == 'W').sum()),
        'Win%': round((group['Grade'] == 'W').mean() * 100, 1),
        'Staked': int(staked),
        'ExpProfit': int(exp_profit),
        'Profit': int(group['Profit'].sum()),
        'ExpROI%': round(exp_profit / staked * 100, 1),
        'ROI%': round(group['Profit'].sum() / staked * 100, 1),
    })

def int_cols(summary):
    for col in ['Bets', 'Wins', 'Staked', 'ExpProfit', 'Profit']:
        summary[col] = summary[col].astype(int)
    return summary

print('=== Performance by Stake Band (Merged) ===')
stake_summary = mdf.groupby('StakeBand', observed=False).apply(summarize, include_groups=False).reset_index()
display(int_cols(stake_summary))

=== Performance by Stake Band (Merged) ===


,StakeBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,0-500,123,59,48.0,34038,4093,4646,12.0,13.6
1,500-1000,83,39,47.0,56934,5954,8460,10.5,14.9
2,1000-2000,50,19,38.0,71535,9700,-2800,13.6,-3.9
3,2000+,47,16,34.0,206031,23576,-24188,11.4,-11.7


In [29]:
# --- Export Performance Tables to CSV ---

blank = pd.DataFrame([{}])

odds_export = odds_summary.rename(columns={'OddsBand': 'Bucket'}).copy()
edge_export = edge_summary.rename(columns={'EdgeBand': 'Bucket'}).copy()
bettype_export = bettype_summary.rename(columns={'BetType': 'Bucket'}).copy()
totals_export = totals.assign(Bucket='All').copy()
stake_export = stake_summary.rename(columns={'StakeBand': 'Bucket'}).copy()

odds_export.insert(0, 'Group', 'Odds Band')
edge_export.insert(0, 'Group', 'Edge Band')
bettype_export.insert(0, 'Group', 'Bet Type')
totals_export.insert(0, 'Group', 'Grand Total')
stake_export.insert(0, 'Group', 'Stake Band')

combined = pd.concat([
    odds_export,
    blank,
    edge_export,
    blank,
    bettype_export,
    blank,
    totals_export,
    blank,
    blank,
    stake_export,
], ignore_index=True)

# Reorder so Group and Bucket are first
cols = ['Group', 'Bucket'] + [c for c in combined.columns if c not in ('Group', 'Bucket')]
combined = combined[cols]

combined.to_csv('performance_summary.csv', index=False)
print(f'Exported to performance_summary.csv')
combined

Exported to performance_summary.csv


,Group,Bucket,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,Odds Band,-150 or less,12.0,7.0,58.3,27209.0,2068.0,-2.0,7.6,-0.0
1,Odds Band,-150 to -100,86.0,50.0,58.1,107363.0,10517.0,-4750.0,9.8,-4.4
2,Odds Band,+100 to +150,290.0,145.0,50.0,197983.0,22633.0,-3230.0,11.4,-1.6
3,Odds Band,+150 or greater,76.0,23.0,30.3,35984.0,6743.0,-5900.0,18.7,-16.4
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Edge Band,0-5%,19.0,11.0,57.9,24857.0,720.0,-4342.0,2.9,-17.5
6,Edge Band,5-10%,139.0,72.0,51.8,127138.0,10534.0,3894.0,8.3,3.1
7,Edge Band,10-15%,164.0,80.0,48.8,125927.0,14964.0,-9614.0,11.9,-7.6
8,Edge Band,15%+,116.0,49.0,42.2,71972.0,16523.0,-1546.0,23.0,-2.1
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# --- Flat Stake Analysis (RichStake = 1 for all bets) ---

fdf = pd.read_csv('merged_data.csv')

# Filter out pushes and mixed grades
fdf = fdf[~fdf['Grade'].isin(['P', 'mix'])].copy()

# Filter out extreme odds: greater than or equal to +300 or less than or equal to -150
fdf = fdf[(fdf['LineTaken'] > -150) & (fdf['LineTaken'] < 300)].copy()

print(f'After filtering: {len(fdf)} bets')
print(f'Grade breakdown: {fdf["Grade"].value_counts().to_dict()}')

# Replace all RichStake with 1
fdf['RichStake'] = 1

# Recalculate ExpProfit with flat stake
fdf['ExpProfit'] = (fdf['RichStake'] * fdf['Edge'] / 100).round(4)

# Recalculate Profit with flat stake
def calc_profit_flat(row):
    if row['Grade'] == 'L':
        return -1.0
    # Grade == W: calculate winnings from american odds
    odds = row['LineTaken']
    if odds > 0:
        return odds / 100
    else:
        return 100 / abs(odds)

fdf['Profit'] = fdf.apply(calc_profit_flat, axis=1)

def summarize_flat(group):
    staked = group['RichStake'].sum()
    exp_profit = group['ExpProfit'].sum()
    return pd.Series({
        'Bets': int(len(group)),
        'Wins': int((group['Grade'] == 'W').sum()),
        'Win%': round((group['Grade'] == 'W').mean() * 100, 1),
        'Staked': int(staked),
        'ExpProfit': round(exp_profit, 2),
        'Profit': round(group['Profit'].sum(), 2),
        'ExpROI%': round(exp_profit / staked * 100, 1),
        'ROI%': round(group['Profit'].sum() / staked * 100, 1),
    })

def int_cols_flat(summary):
    for col in ['Bets', 'Wins', 'Staked']:
        summary[col] = summary[col].astype(int)
    return summary

# --- 1. By Odds Bands ---
odds_bins = [-150, -100, 150, 300]
odds_labels = ['-150 to -100', '+100 to +150', '+151 to +300']
fdf['OddsBand'] = pd.cut(fdf['LineTaken'], bins=odds_bins, labels=odds_labels)

print('\n=== Flat Stake: Performance by Odds Band ===')
f_odds = fdf.groupby('OddsBand', observed=False).apply(summarize_flat, include_groups=False).reset_index()
display(int_cols_flat(f_odds))

# --- 2. By Edge Bands ---
edge_bins = [0, 5, 10, 15, float('inf')]
edge_labels = ['0-5%', '5-10%', '10-15%', '15%+']
fdf['EdgeBand'] = pd.cut(fdf['Edge'], bins=edge_bins, labels=edge_labels)

print('\n=== Flat Stake: Performance by Edge Band ===')
f_edge = fdf.groupby('EdgeBand', observed=False).apply(summarize_flat, include_groups=False).reset_index()
display(int_cols_flat(f_edge))

# --- 3. By BetType ---
print('\n=== Flat Stake: Performance by BetType ===')
core_types = fdf[fdf['BetType'].isin(['moneyline', 'spread', 'total'])]
f_bettype = core_types.groupby('BetType').apply(summarize_flat, include_groups=False).reset_index()
display(int_cols_flat(f_bettype))

# --- 4. Grand Totals ---
print('\n=== Flat Stake: Grand Totals ===')
f_totals = summarize_flat(fdf).to_frame().T
display(int_cols_flat(f_totals))

# --- Export to CSV ---
blank = pd.DataFrame([{}])

f_odds_ex = f_odds.rename(columns={'OddsBand': 'Bucket'}).copy()
f_edge_ex = f_edge.rename(columns={'EdgeBand': 'Bucket'}).copy()
f_bettype_ex = f_bettype.rename(columns={'BetType': 'Bucket'}).copy()
f_totals_ex = f_totals.assign(Bucket='All').copy()

f_odds_ex.insert(0, 'Group', 'Odds Band')
f_edge_ex.insert(0, 'Group', 'Edge Band')
f_bettype_ex.insert(0, 'Group', 'Bet Type')
f_totals_ex.insert(0, 'Group', 'Grand Total')

combined_flat = pd.concat([
    f_odds_ex, blank, f_edge_ex, blank, f_bettype_ex, blank, f_totals_ex
], ignore_index=True)

cols = ['Group', 'Bucket'] + [c for c in combined_flat.columns if c not in ('Group', 'Bucket')]
combined_flat = combined_flat[cols]

combined_flat.to_csv('flat_stake_performance.csv', index=False)
print('\nExported to flat_stake_performance.csv')
combined_flat

After filtering: 261 bets
Grade breakdown: {'L': 134, 'W': 127}

=== Flat Stake: Performance by Odds Band ===


,OddsBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,-150 to -100,46,26,56.5,46,4.05,2.51,8.8,5.5
1,+100 to +150,181,88,48.6,181,18.74,11.36,10.4,6.3
2,+151 to +300,34,13,38.2,34,4.91,4.65,14.4,13.7



=== Flat Stake: Performance by Edge Band ===


,EdgeBand,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,0-5%,18,10,55.6,18,0.62,3.74,3.4,20.8
1,5-10%,83,43,51.8,83,7.08,8.98,8.5,10.8
2,10-15%,97,52,53.6,97,11.72,19.10,12.1,19.7
3,15%+,45,13,28.9,45,9.10,-14.61,20.2,-32.5



=== Flat Stake: Performance by BetType ===


,BetType,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,moneyline,58,31,53.4,58,6.91,12.37,11.9,21.3
1,spread,110,49,44.5,110,11.69,-4.66,10.6,-4.2
2,total,91,47,51.6,91,8.74,12.81,9.6,14.1



=== Flat Stake: Grand Totals ===


,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,261,127,48.7,261,27.7,18.52,10.6,7.1



Exported to flat_stake_performance.csv


,Group,Bucket,Bets,Wins,Win%,Staked,ExpProfit,Profit,ExpROI%,ROI%
0,Odds Band,-150 to -100,46.0,26.0,56.5,46.0,4.05,2.51,8.8,5.5
1,Odds Band,+100 to +150,181.0,88.0,48.6,181.0,18.74,11.36,10.4,6.3
2,Odds Band,+151 to +300,34.0,13.0,38.2,34.0,4.91,4.65,14.4,13.7
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Edge Band,0-5%,18.0,10.0,55.6,18.0,0.62,3.74,3.4,20.8
5,Edge Band,5-10%,83.0,43.0,51.8,83.0,7.08,8.98,8.5,10.8
6,Edge Band,10-15%,97.0,52.0,53.6,97.0,11.72,19.10,12.1,19.7
7,Edge Band,15%+,45.0,13.0,28.9,45.0,9.10,-14.61,20.2,-32.5
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Bet Type,moneyline,58.0,31.0,53.4,58.0,6.91,12.37,11.9,21.3
